## Simple RAG with Additional Tools
Here we can use the Python SDK to develop a RAG agent with Milvus retrievers, web search, and code generation tools.

**Prerequisites:**
- Milvus server running at `localhost:19530`
- Collections named `cuda_docs` and `mcp_docs` with embedded documents
- TAVILY_API_KEY environment variable set for web search
- `nvidia-nat[langchain]` package installed for Tavily and CodeGeneration tools


In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath("../../../src/")
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)


In [ ]:
from pydantic import HttpUrl

from nat.agent.react_agent.register import NatReActAgent
from nat.embedder.nim_embedder import NIMEmbedder
from nat.llm.nim_llm import NimLLM
from nat.plugins.langchain.tools.code_generation_tool import CodeGenerationTool
from nat.plugins.langchain.tools.tavily_internet_search import TavilyInternetSearchTool
from nat.retriever.milvus.register import MilvusRetriever
from nat.tool.retriever import NatRetrieverTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0,
    max_tokens=4096,
    top_p=1.0,
    name="nim_llm",
)

milvus_embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    truncate="END",
    name="milvus_embedder",
)

cuda_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="cuda_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="cuda_retriever",
)

mcp_retriever = MilvusRetriever(
    uri=HttpUrl("http://localhost:19530"),
    collection_name="mcp_docs",
    embedder=milvus_embedder,
    top_k=10,
    name="mcp_retriever",
)

cuda_retriever_tool = NatRetrieverTool(
    nat_retriever=cuda_retriever,
    topic="Retrieve documentation for NVIDIA's CUDA library",
    name="cuda_retriever_tool",
)

mcp_retriever_tool = NatRetrieverTool(
    nat_retriever=mcp_retriever,
    topic="Retrieve information about Model Context Protocol (MCP)",
    name="mcp_retriever_tool",
)

# Web search tool - requires TAVILY_API_KEY environment variable
web_search_tool = TavilyInternetSearchTool(
    max_results=5,
    name="web_search_tool",
)

# Code generation tool
code_generation_tool = CodeGenerationTool(
    llm=llm,
    description=(
        "Always call this tool to generate python code. "
        "Returns a code snippet which MUST be included in your response."
    ),
    name="code_generation_tool",
)

agent = NatReActAgent(
    tools=[cuda_retriever_tool, mcp_retriever_tool, web_search_tool, code_generation_tool],
    llm=llm,
    verbose=True,
    additional_instructions=(
        "If a tool call results in code or other artifacts being returned, "
        "you MUST include that in your thoughts and response."
    ),
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)


In [ ]:
await nat_workflow.prompt('Write Python code to install CUDA using pip')


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config_tools.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())
